In [2]:
import requests, tarfile, io, os, rdata, pandas as pd

url = "https://cran.r-project.org/src/contrib/visualFields_1.0.7.tar.gz"
print("Downloading visualFields tarball...")
r = requests.get(url)
tar = tarfile.open(fileobj=io.BytesIO(r.content), mode="r:gz")

out_dir = r"C:\Users\mm3572\PyVisualField\src\PyVisualFields\data"
os.makedirs(out_dir, exist_ok=True)

datasets = [
    "vfpwgRetest24d2", "vfctrSunyiu24d2", "vfctrSunyiu10d2",
    "vfpwgSunyiu24d2", "vfctrIowaPC26",   "vfctrIowaPeri",
]

for name in datasets:
    member_path = f"visualFields/data/{name}.rda"
    try:
        f = tar.extractfile(member_path)
        raw_bytes = f.read()                          # <-- read as bytes first
        parsed = rdata.parser.parse_data(raw_bytes)   # <-- pass bytes directly
        converted = rdata.conversion.convert(parsed)
        df = converted[name]
        df.to_csv(os.path.join(out_dir, f"{name}.csv"), index=False)
        print(f"  OK  {name}.csv  {df.shape}")
    except Exception as e:
        print(f"  FAIL  {name}: {e}")

print("Done.")

C:\Users\mm3572\AppData\Local\anaconda3\envs\env_pyVF\lib\site-packages\rdata\conversion\_conversion.py:843: UserWarning: Missing constructor for R class "Date". The underlying R object is returned instead.
  warnings.warn(
C:\Users\mm3572\AppData\Local\anaconda3\envs\env_pyVF\lib\site-packages\rdata\conversion\_conversion.py:843: UserWarning: Missing constructor for R class "Date". The underlying R object is returned instead.
  warnings.warn(
C:\Users\mm3572\AppData\Local\anaconda3\envs\env_pyVF\lib\site-packages\rdata\conversion\_conversion.py:843: UserWarning: Missing constructor for R class "Date". The underlying R object is returned instead.
  warnings.warn(
C:\Users\mm3572\AppData\Local\anaconda3\envs\env_pyVF\lib\site-packages\rdata\conversion\_conversion.py:843: UserWarning: Missing constructor for R class "Date". The underlying R object is returned instead.
  warnings.warn(
C:\Users\mm3572\AppData\Local\anaconda3\envs\env_pyVF\lib\site-packages\rdata\conversion\_conversion.py:

  OK  vfpwgRetest24d2.csv  (360, 64)
  OK  vfctrSunyiu24d2.csv  (263, 64)
  OK  vfctrSunyiu10d2.csv  (55, 78)
  OK  vfpwgSunyiu24d2.csv  (42, 64)
  OK  vfctrIowaPC26.csv  (196, 81)
  OK  vfctrIowaPeri.csv  (196, 80)
Done.


In [4]:
import requests, tarfile, io, os, rdata, pandas as pd

url = "https://cran.r-project.org/src/contrib/vfprogression_0.7.1.tar.gz"
print("Downloading vfprogression tarball...")
r = requests.get(url)
tar = tarfile.open(fileobj=io.BytesIO(r.content), mode="r:gz")

out_dir = r"C:\Users\mm3572\PyVisualField\src\PyVisualFields\data"
os.makedirs(out_dir, exist_ok=True)

datasets = ["vfseries", "vf.vfi", "vf.cigts", "vf.plr.nouri.2012", "vf.schell2014"]
extensions = [".RData", ".rda"]  # try both

for name in datasets:
    saved = False
    for ext in extensions:
        member_path = f"vfprogression/data/{name}{ext}"
        try:
            f = tar.extractfile(member_path)
            if f is None:
                continue
            raw_bytes = f.read()
            parsed = rdata.parser.parse_data(raw_bytes)
            converted = rdata.conversion.convert(parsed)
            key = name if name in converted else list(converted.keys())[0]
            df = converted[key]
            df.to_csv(os.path.join(out_dir, f"{name}.csv"), index=False)
            print(f"  OK  {name}.csv  {df.shape}")
            saved = True
            break
        except Exception as e:
            continue
    if not saved:
        print(f"  FAIL  {name}: not found with any extension")

print("Done.")

  OK  vfseries.csv  (20, 278)
  OK  vf.vfi.csv  (20, 3)
  OK  vf.cigts.csv  (20, 54)
  OK  vf.plr.nouri.2012.csv  (20, 54)
  OK  vf.schell2014.csv  (20, 2)
Done.


In [14]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

# Load the control dataset
df = pd.read_csv(r"C:\Users\mm3572\PyVisualField\src\PyVisualFields\data\vfctrSunyiu24d2.csv")

# Sensitivity columns are l1..l54
scols = sorted([c for c in df.columns if c.startswith('l') and c[1:].isdigit()],
               key=lambda x: int(x[1:]))
print(f"Found {len(scols)} sensitivity columns: {scols[:5]}...")

ages = df['age'].values.astype(float)

intercepts, slopes = [], []
for col in scols:
    y = df[col].values.astype(float)
    mask = ~np.isnan(y)
    sl, ic, *_ = linregress(ages[mask], y[mask])
    intercepts.append(ic)
    slopes.append(sl)

intercepts = np.array(intercepts)
slopes     = np.array(slopes)

print("\nIntercepts:\n", np.round(intercepts, 4))
print("\nSlopes:\n",     np.round(slopes, 6))

# Save
out = r"C:\Users\mm3572\PyVisualField\src\PyVisualFields\data\normvals_sunyiu24d2.csv"
pd.DataFrame({'intercept': intercepts, 'slope': slopes}).to_csv(out, index=False)
print(f"\nSaved to {out}")

Found 54 sensitivity columns: ['l1', 'l2', 'l3', 'l4', 'l5']...

Intercepts:
 [32.5074 31.5454 29.4486 30.4795 32.8119 33.0642 33.3731 32.1717 33.5396
 33.5321 33.4273 33.9652 34.6535 34.7405 34.8169 34.3631 34.58   34.1394
 32.8288 33.9983 34.8411 35.5256 35.6608 35.0065 35.5912 29.4116 33.6027
 33.0361 33.7765 35.4161 35.3313 35.7622 36.122  36.7303 -0.2368 34.2653
 33.3825 34.3298 35.1684 34.9852 34.9135 36.6976 34.8736 33.7237 34.1363
 35.4714 34.0548 34.1519 33.9896 35.4636 34.8882 34.4029 34.0784 34.7127]

Slopes:
 [-0.095664 -0.077102 -0.056238 -0.075298 -0.076123 -0.06416  -0.068661
 -0.057969 -0.076331 -0.087089 -0.083613 -0.062191 -0.056784 -0.058252
 -0.067969 -0.074093 -0.081165 -0.088178 -0.099322 -0.081595 -0.061475
 -0.055208 -0.047067 -0.041285 -0.070834 -0.060802 -0.064014 -0.101382
 -0.071802 -0.062658 -0.045674 -0.045695 -0.051634 -0.079414  0.054965
 -0.070606 -0.079263 -0.054993 -0.049742 -0.051327 -0.051829 -0.086138
 -0.066364 -0.065601 -0.070981 -0.079471 -0.056